> **🎯 Important**
>
**Durée estimée** : 10 à 12 heures  
**Prérequis** : toutes les notions du Module 7 (7.1 à 7.6, **sauf 7.6 optionnel**)  
**Objectif** : construire un chatbot professionnel combinant RAG, agent, outils métier et logs


> **💡 Astuce**
>
## 💼 À la fin, tu auras un vrai projet à montrer
Ce TP produit un **chatbot fonctionnel** que tu peux :
- Montrer à un recruteur
- Adapter à ton propre domaine (ta boîte, ton blog, un wiki...)
- Mettre sur ton GitHub
- Décrire dans ton CV comme projet "IA générative / RAG / Agents"


# 🎯 Contexte

Tu es data scientist freelance. **TechVolt**, la startup de bornes de recharge électrique dont tu as entendu parler pendant tout ce module, te contacte :

> *« Notre équipe support est débordée. Les techniciens sur le terrain posent toujours les mêmes questions : codes d'erreur, horaires SAV, prix des abonnements, procédures d'installation... On voudrait un chatbot qui connaît TOUTE notre doc et peut même **agir** (créer un ticket, prendre un RDV). Budget : 0€ (startup !). Livrable : un prototype fonctionnel en 2 semaines. Go ? »*

## 🎁 Les objectifs business

1. **Répondre** automatiquement aux questions sur la doc (RAG)
2. **Diagnostiquer** les codes d'erreur
3. **Créer des tickets** pour les cas non résolus
4. **Prendre des RDV** d'installation
5. **Tout gratuit** (Groq + ChromaDB + sentence-transformers)
6. **Logs** pour que l'équipe suive ce qui se passe
7. **Évaluer** la qualité des réponses (bonus)

## 📦 Ce qu'on te fournit

Un dossier `ressources_tp/` avec :
- **`knowledge_base.json`** : 26 documents TechVolt (installation, erreurs, SAV, commercial, sécurité)
- **`test_questions.json`** : 10 questions de test pour l'évaluation

## 🗺️ Plan du TP

Le TP se structure en **7 parties** :

1. **Setup et exploration des données**
2. **Préparer le RAG** (embeddings + ChromaDB)
3. **Prompt RAG et première conversation**
4. **Outils métier** (diagnostic, tickets, RDV)
5. **L'agent complet** (RAG + outils)
6. **Conversation multi-tours** avec mémoire
7. **Bonus** : évaluation automatique avec LLM-as-judge

# 🛠️ Mise en route

In [ ]:
import os
import json
import time
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date, datetime
from typing import Literal
from dataclasses import dataclass

# Logging propre
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("techvolt")

print("✅ Environnement prêt")

> **🎯 Important**
>
## 📦 Librairies nécessaires + clé API

```bash
pip install sentence-transformers chromadb openai pydantic python-dotenv
```

**N'oublie pas ta clé Groq** dans `.env` :
```
GROQ_API_KEY=gsk_ta_cle_ici
```

In [ ]:
# Chemin vers les ressources
DATA_DIR = Path("ressources_tp")

In [ ]:
DATA_DIR = Path("ressources_tp")

---

# Partie 1 — Setup et exploration

## ✏️ Étape 1.1 — Charger le corpus

> **ℹ️ Note**
>
## 📝 À faire

1. Charge le fichier JSON `ressources_tp/knowledge_base.json`
2. Affiche le **nombre total** de documents
3. Affiche la **répartition par catégorie** (valeur `categorie`)
4. Affiche le **premier document** de chaque catégorie pour avoir une idée du contenu

In [ ]:
# TODO: Étape 1.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 1.2 — Statistiques sur le corpus

> **ℹ️ Note**
>
## 📝 À faire

Calcule et affiche :
1. **Longueur moyenne** des documents (en caractères et en mots)
2. **Document le plus long** et **le plus court** (titre + longueur)
3. Un **barplot** de la répartition par catégorie

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 2 — Préparer le RAG

## ✏️ Étape 2.1 — Générer les embeddings

> **ℹ️ Note**
>
## 📝 À faire

1. Charge le modèle `paraphrase-multilingual-mpnet-base-v2` (modèle multilingue, bon en français)
2. Encode **tous les contenus** du corpus en **batch** (une seule fois, plus rapide que en boucle)
3. Affiche la **shape** des embeddings (devrait être `(26, 768)`)
4. **Bonus** : mesure le temps d'encoding (pour savoir combien ça coûte)

In [ ]:
# TODO: Étape 2.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 2.2 — Indexer dans ChromaDB

> **ℹ️ Note**
>
## 📝 À faire

1. Crée un client **persistant** ChromaDB (path `./chroma_techvolt`)
2. Crée une collection `techvolt_kb` avec **métrique cosine** (`hnsw:space: cosine`)
3. Si la collection existe déjà, **supprime-la** avant pour repartir propre
4. Ajoute tous les documents avec :
   - `ids` : les ID du corpus (INST-001, ERR-E01...)
   - `documents` : les contenus
   - `embeddings` : les embeddings précalculés
   - `metadatas` : un dict avec `titre` et `categorie`
5. Vérifie le `collection.count()`

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 2.3 — Tester la recherche

> **ℹ️ Note**
>
## 📝 À faire

Écris une fonction `rechercher(question: str, k: int = 3, filtre_categorie: str = None) -> list` qui :
1. Encode la question
2. Query ChromaDB avec `n_results=k`
3. Si `filtre_categorie` est fourni, filtre via `where={"categorie": ...}`
4. Retourne une liste de dicts `{id, titre, contenu, categorie, distance}`

Teste-la sur 3 questions :
- "Comment installer ma borne ?"
- "C'est quoi le code E04 ?"
- "Combien coûte l'abonnement ?" (avec filtre `commercial`)

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 3 — Le prompt RAG et la première conversation

## ✏️ Étape 3.1 — Template de prompt RAG strict

> **ℹ️ Note**
>
## 📝 À faire

Écris la constante `PROMPT_RAG` (template Python avec `{sources}` et `{question}`) qui :
1. Positionne le LLM comme **assistant support TechVolt**
2. Impose des **règles strictes** : sources uniquement, citations [Source N], "je ne sais pas"
3. Fournit les sources formatées
4. Prévoit un ton **professionnel mais accessible**

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 3.2 — Fonction rag_repondre

> **ℹ️ Note**
>
## 📝 À faire

Écris une fonction `rag_repondre(question: str, k: int = 3) -> dict` qui :
1. Utilise `rechercher()` pour obtenir les k passages pertinents
2. Formate les sources proprement pour le prompt
3. Appelle le LLM Groq (Llama 3.3 70B) avec `temperature=0`
4. Retourne `{"reponse": ..., "passages": [...], "prompt_utilise": ...}`

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 3.3 — Tester sur 5 questions variées

> **ℹ️ Note**
>
## 📝 À faire

Teste ton RAG sur ces 5 questions, en affichant pour chaque question :
- La réponse du LLM
- Les sources utilisées

```python
questions = [
    "C'est quoi le code E04 ?",
    "Combien coûte la borne Pro ?",
    "Est-ce que mon installation est éligible aux aides ?",
    "Ma borne ne capte plus le wifi depuis ce matin",
    "Quelle est la meilleure pizza italienne ?",  # hors sujet !
]
```

La question hors sujet doit déclencher le "Je ne trouve pas cette information..." du prompt.

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 4 — Outils métier

Le RAG c'est bien, mais il ne peut pas **agir**. Ajoutons des outils métier.

## ✏️ Étape 4.1 — Définir 4 fonctions outils

> **ℹ️ Note**
>
## 📝 À faire

Écris **4 fonctions Python** :

1. **`diagnostic_erreur(code: str) -> dict`** — retourne un dict avec `code`, `niveau_urgence`, `description`, `action`. Utilise les documents ERR-E01 à ERR-E05 pour extraire les infos.

2. **`creer_ticket_support(probleme: str, urgence: str) -> str`** — crée un ticket simulé. Retourne une chaîne genre `"Ticket TVT-12345 créé (urgence: URGENTE). Un technicien vous contactera sous 24h."`

3. **`prendre_rdv_installation(date_iso: str, creneau: str) -> str`** — prend un RDV. Valide la date (pas dans le passé) et le créneau (matin/après-midi). Retourne une confirmation avec référence.

4. **`get_horaires_sav(jour: str) -> str`** — retourne les horaires du SAV pour un jour. Gère les jours fériés et dimanche (fermé).

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 4.2 — Schémas JSON pour le LLM

> **ℹ️ Note**
>
## 📝 À faire

Écris la liste `OUTILS_SCHEMAS` (liste de dicts au format function calling OpenAI) qui décrit **précisément** chaque outil au LLM. N'oublie pas :
- `description` qui dit clairement **quand** utiliser l'outil
- `parameters` avec types, descriptions et `required`

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 5 — L'agent complet (RAG + outils)

Maintenant on combine tout : l'agent a accès à **la doc (via RAG)** ET aux **outils métier**.

## 🎯 Architecture de l'agent

L'agent a **2 capacités** :
1. **Chercher dans la doc** via un outil `rechercher_doc` (wrapper autour du RAG)
2. **Utiliser les outils métier** (diagnostic, ticket, RDV, horaires)

Il décide lui-même quand utiliser quoi.

## ✏️ Étape 5.1 — Wrapper RAG comme outil

> **ℹ️ Note**
>
## 📝 À faire

1. Écris une fonction `rechercher_doc(question: str) -> str` qui :
   - Utilise la fonction `rechercher()` avec k=3
   - Retourne les 3 passages concaténés avec leurs titres
2. Ajoute le schéma à `OUTILS_SCHEMAS`
3. Ajoute la fonction à `OUTILS_FONCTIONS`

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 5.2 — System prompt de l'agent

> **ℹ️ Note**
>
## 📝 À faire

Écris une constante `SYSTEM_AGENT` qui fait de ton LLM un **agent support TechVolt** :
- Persona claire
- Liste des outils à utiliser
- Cas d'urgence (E04 → ticket URGENT **obligatoire**)
- Ton pro mais accessible

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 5.3 — Boucle de l'agent

> **ℹ️ Note**
>
## 📝 À faire

Écris une fonction `agent_techvolt(question: str, max_iter: int = 6) -> dict` qui :
1. Initialise les messages avec le system prompt + la question utilisateur
2. Boucle jusqu'à `max_iter` :
   - Appelle le LLM avec les outils disponibles
   - Si pas de tool call → retourne la réponse finale
   - Sinon : exécute les outils, ajoute les résultats, continue
3. Détecte les boucles (3 fois le même appel → stop)
4. Retourne un dict avec `reponse`, `appels_outils`, `nb_iterations`, `duree`

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 5.4 — Scénarios de test (démo scripted)

> **ℹ️ Note**
>
## 📝 À faire

Teste ton agent sur **5 scénarios clients** variés et réalistes :

```python
scenarios = [
    # Simple : RAG seul
    "Est-ce que je peux recharger ma Tesla Model 3 avec une borne TechVolt ?",
    # Outil simple
    "Le SAV est ouvert dimanche ?",
    # RAG + diagnostic
    "Ma borne affiche E03, que faire ?",
    # Multi-outils (cas critique !)
    "Ma borne fait des bruits bizarres et affiche E04, c'est grave ? "
    "Ouvrez-moi un ticket urgent et dites-moi si le SAV est joignable maintenant.",
    # Question composée
    "Je veux installer une borne Pro à mon domicile. "
    "Combien ça coûte, y a-t-il des aides, et vous pouvez réserver "
    "un créneau d'installation pour le 20 juillet 2026 matin ?",
]
```

Pour chaque scénario :
- Affiche la question et la réponse
- Affiche la **liste des outils appelés**
- Vérifie les **résultats attendus** (ticket urgent sur E04, RDV pris, etc.)

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 6 — Conversation multi-tours avec mémoire

Jusqu'ici, chaque question est traitée **indépendamment**. Un vrai chatbot **se souvient** de l'échange.

## ✏️ Étape 6.1 — Classe Chatbot avec historique

> **ℹ️ Note**
>
## 📝 À faire

Écris une classe `ChatbotTechVolt` avec :
- Attribut `historique` : liste des messages échangés
- Méthode `poser_question(question: str) -> str` : appelle l'agent avec **l'historique complet**
- Méthode `reset()` : vide l'historique
- Méthode `resumer_conversation() -> dict` : retourne stats (nb messages, outils totaux...)

**Astuce** : pour garder l'agent capable de nouveaux tool calls à chaque tour, passe l'historique dans `messages` **avant** le message user.

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 7 — BONUS : évaluation automatique

Cette partie est **facultative mais très formatrice**. Elle te donne une **compétence pro** très recherchée.

## 🎯 Objectif

Évaluer automatiquement la qualité du RAG en utilisant un LLM comme juge (LLM-as-judge).

## ✏️ Étape 7.1 (bonus) — Dataset d'évaluation

> **ℹ️ Note**
>
## 📝 À faire

Crée un dataset de **10 questions de test** avec pour chaque question :
- La question
- La **réponse attendue** (résumé court)
- Les **ID des sources** qu'on devrait trouver
- Une **catégorie** : factuel, procédural, hors-sujet

Sauvegarde dans `ressources_tp/test_questions.json`.

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 7.2 (bonus) — Juge LLM

> **ℹ️ Note**
>
## 📝 À faire

Écris une fonction `evaluer_reponse(question: str, reponse_obtenue: str, reponse_attendue: str) -> dict` qui :
1. Envoie au LLM un prompt demandant de **noter** la réponse obtenue de 1 à 5
2. Critères : pertinence, exactitude, fidélité aux sources
3. Retourne `{"note": int, "justification": str}`

Puis applique cette évaluation sur les 10 questions et calcule la **note moyenne**.

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# 🏁 Bilan et pour aller plus loin

## 🏆 Ce que tu as accompli

Tu viens de construire un **chatbot professionnel complet** :

- ✅ **Corpus réaliste** (26 docs TechVolt structurés)
- ✅ **RAG avec ChromaDB persistant** et filtres catégorie
- ✅ **Prompt RAG strict** anti-hallucinations
- ✅ **5 outils métier** (diagnostic, ticket, RDV, horaires, recherche)
- ✅ **Agent multi-tours** qui enchaîne outils intelligemment
- ✅ **Détection de boucles** pour la sécurité
- ✅ **Conversation avec mémoire**
- ✅ **Évaluation automatique** par LLM-as-judge (bonus)
- ✅ **Logs structurés** pour monitoring

**Tu combines toutes les briques** que tu as apprises dans le Module 7. 🎉

## 💼 Pour aller plus loin (projet perso)

Voici des pistes pour **transformer ce TP** en projet portefeuille CV :

1. **Adapter à TES données** — ta doc de boulot, ton blog, un wiki...
2. **Passer en architecture fichiers** — séparer `src/agent.py`, `src/tools.py`, `chatbot.py`...
3. **Interface web** avec Gradio ou Streamlit (50 lignes)
4. **Déploiement** — Hugging Face Spaces (gratuit), Railway, ou VPS
5. **Guardrails** — filtrer les questions hors-sujet avant l'agent
6. **Cache** — mémoriser les réponses fréquentes
7. **Monitoring** — logs structurés vers Sentry ou Datadog
8. **CI/CD** — tests sur le dataset d'éval à chaque push

## 📊 Métriques business attendues

Pour un vrai chatbot support :
- **Taux de résolution autonome** (sans humain) : viser > 60%
- **Temps de réponse** : < 5s (ton RAG est à ~2-3s)
- **Coût par requête** : ~0€ avec Groq (free tier) 🎉
- **Faithfulness** (LLM-as-judge) : viser > 4/5

## 🎓 Module 7 : tu as terminé !

Tu maîtrises maintenant la **boîte à outils complète de l'IA générative** :

- ✅ Embeddings (7.1)
- ✅ Transformer (7.2)
- ✅ API LLM (7.3)
- ✅ RAG (7.4)
- ✅ Agents (7.5)
- ✅ Fine-tuning (7.6)
- ✅ **Projet complet** (ce TP)

Tu es à **niveau junior confirmé / intermédiaire** en IA générative. Suffisant pour décrocher un poste d'IA Engineer en 2026.

## 🚀 La suite : le Module 8 — MLOps

Il te reste **un dernier module** pour parfaire ton arsenal : le **MLOps** (déploiement, monitoring, CI/CD pour les modèles ML).

C'est la **discipline** qui fait la différence entre :
- Un script qui marche sur ta machine
- Un **système** qui tourne en production, se met à jour, se surveille, et résiste aux pannes

Ton profil sera alors **complet**. 🎯

> **💡 Astuce**
>
## 💡 Défi final pour ce module

**Déploie ton chatbot sur Hugging Face Spaces** (gratuit) pour qu'il soit accessible par URL :

1. Crée un compte sur [huggingface.co](https://huggingface.co)
2. New Space → Gradio → Free GPU
3. Uploade tes fichiers + un `app.py` avec une interface Gradio minimale
4. Ajoute ta clé API dans les secrets du Space
5. Ton chatbot est **en ligne** 🎉

Tu peux partager l'URL à des amis, des recruteurs, et le mettre dans ta présentation.